# Aula 22 — Modelagem de tópicos

Na aula anterior transformamos notícia em texto preparado. Hoje vamos perguntar: **que assuntos aparecem sem informar antes as cinco editorias?** Um tópico é um conjunto de palavras que costuma aparecer junto; o nome que damos a ele é interpretação humana.

**Roteiro:** carregar → preparar → TF-IDF → NMF e LDA → ler notícias representativas → comparar com editorias → experimentar BERTopic. Abra no Google Colab. A parte principal roda sem GPU; a última usa GPU.

## 1. Corpus e revisão

Use a mesma fonte da Aula 21. A coluna `editoria` serve para conferir o resultado depois; ela **não entra** no treinamento. Textos duplicados e vazios devem sair da matriz, com contagem registrada.

In [1]:
import io
import zipfile
import requests
import pandas as pd

url = "https://codeload.github.com/mauricio-seiji/Dataset-news-articles-pt-br/zip/refs/heads/main"
resposta = requests.get(url, timeout=40)
resposta.raise_for_status()
zip_fonte = zipfile.ZipFile(io.BytesIO(resposta.content))
linhas = []
for caminho in zip_fonte.namelist():
    if caminho.endswith(".txt") and "__summarization__" not in caminho:
        partes = caminho.split("/")
        linhas.append({"id": partes[-2] + "/" + partes[-1],
                       "editoria": partes[-2],
                       "texto_original": zip_fonte.read(caminho).decode("utf-8")})
noticias = pd.DataFrame(linhas).sort_values("id").reset_index(drop=True)
print(noticias.shape)
print(noticias["editoria"].value_counts())
assert len(noticias) == 500

(500, 3)
editoria
esportes                103
ciencia e tecnologia    100
entretenimento          100
politica                100
economia                 97
Name: count, dtype: int64


In [2]:
import re
base = noticias.drop_duplicates(subset="texto_original").copy()
base = base[base["texto_original"].str.len() >= 300].reset_index(drop=True)
base["texto_modelo"] = base["texto_original"].str.lower().map(
    lambda s: " ".join(re.findall(r"\b[^\W\d_]+\b", s, flags=re.UNICODE))
)
print("Textos usados:", len(base), "de", len(noticias))
print(base[["id", "editoria"]].head())

Textos usados: 498 de 500
                             id              editoria
0    ciencia e tecnologia/1.txt  ciencia e tecnologia
1   ciencia e tecnologia/10.txt  ciencia e tecnologia
2  ciencia e tecnologia/100.txt  ciencia e tecnologia
3   ciencia e tecnologia/11.txt  ciencia e tecnologia
4   ciencia e tecnologia/12.txt  ciencia e tecnologia


## 2. Matriz termo-documento e TF-IDF

A matriz tem uma linha por notícia e uma coluna por termo. TF-IDF reduz o peso de palavras presentes em quase todas as notícias. `min_df=3` ignora termos raríssimos; `max_df=0.7` ignora os quase universais. As stopwords abaixo são pequenas e visíveis. Trocar a lista pode mudar os tópicos.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
stopwords = "a o e de da do das dos em no na nos nas um uma para por com que se os as ao foi ser são mais muito também após entre sobre esse essa deste desta seu sua seus suas".split()
import unicodedata
stopwords = [unicodedata.normalize("NFD", w).encode("ascii", "ignore").decode("ascii") for w in stopwords]
vetor = TfidfVectorizer(strip_accents="unicode", stop_words=stopwords,
                        min_df=3, max_df=0.7, max_features=4000)
X = vetor.fit_transform(base["texto_modelo"])
print("Documentos x termos:", X.shape)
print("Termos de exemplo:", vetor.get_feature_names_out()[:15])

Documentos x termos: (498, 4000)
Termos de exemplo: ['abaixo' 'abastecimento' 'abel' 'abertas' 'aberto' 'abertura' 'abordagem'
 'aborto' 'abre' 'abril' 'abrir' 'abriu' 'acaba' 'acabar' 'acabaram']


## 3. NMF: palavras que caminham juntas

NMF decompõe a matriz em cinco combinações de palavras. Cinco é um ponto de partida porque conhecemos cinco editorias; não é um número verdadeiro escondido nos dados. Veja as palavras e depois abra matérias de maior peso em cada tópico.

In [4]:
from sklearn.decomposition import NMF
import numpy as np
nmf = NMF(n_components=5, init="nndsvda", random_state=42, max_iter=300)
pesos = nmf.fit_transform(X)
termos = vetor.get_feature_names_out()
for i, componente in enumerate(nmf.components_):
    print(f"Tópico {i}:", ", ".join(termos[j] for j in componente.argsort()[-10:][::-1]))
base["topico_nmf"] = pesos.argmax(axis=1)
for i in range(5):
    posicoes = pesos[:, i].argsort()[-2:][::-1]
    print("\nTópico", i)
    for pos in posicoes:
        print(base.loc[pos, "editoria"], "—", base.loc[pos, "texto_original"].splitlines()[0][:110])

Tópico 0: como, agua, pode, mas, ou, terra, anos, sistema, pesquisadores, diz
Tópico 1: economia, trimestre, pib, queda, ano, alta, indice, pontos, mercado, pandemia
Tópico 2: bolsonaro, presidente, mourao, trump, governo, eleitoral, biden, ministro, eleicoes, federal
Tópico 3: time, clube, copa, jogos, tecnico, jogo, campeonato, flamengo, final, temporada
Tópico 4: eu, ela, me, voce, ele, gente, minha, meu, album, pra

Tópico 0
ciencia e tecnologia — 20 anos da Estação Espacial Internacional: 6 avanços que o questionado projeto trouxe
ciencia e tecnologia — O telescópio capaz de ver coisas que ocorreram há 13 bilhões de anos

Tópico 1
economia — Governo reduz de 4,7% para 4,5% projeção de queda do PIB em 2020
politica — Economistas do mercado elevam estimativa de inflação para 3,20% em 2020

Tópico 2
politica — Desgaste, 2022 e mais: por que Bolsonaro 'escanteou' Mourão?
politica — 

Tópico 3
esportes — 
esportes — São Paulo não deve encarar Fortaleza com time reserva: "Jogadores sabe

**O que observar:** alguns tópicos lembram uma editoria; outros misturam assuntos ou capturam palavras de um episódio específico. O algoritmo não sabe que `0` significa economia ou que `3` significa esporte. Dê nomes só depois de ler as matérias.

## 4. LDA: outra hipótese sobre os mesmos textos

LDA usa contagens de palavras, não TF-IDF. Vamos comparar as dez palavras principais e perguntar se a interpretação muda. Não escolha o método apenas porque uma lista ficou bonita.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
contagem = CountVectorizer(strip_accents="unicode", stop_words=stopwords,
                           min_df=3, max_df=0.7, max_features=4000)
X_contagem = contagem.fit_transform(base["texto_modelo"])
lda = LatentDirichletAllocation(n_components=5, random_state=42,
                               learning_method="batch", max_iter=10)
pesos_lda = lda.fit_transform(X_contagem)
termos_lda = contagem.get_feature_names_out()
for i, componente in enumerate(lda.components_):
    print(f"LDA {i}:", ", ".join(termos_lda[j] for j in componente.argsort()[-10:][::-1]))

LDA 0: pontos, time, jogos, feira, brasil, pelo, clube, contra, mas, jogo
LDA 1: como, ele, eu, mas, tem, ela, anos, ja, esta, quando
LDA 2: ano, pelo, governo, economia, pela, tem, feira, pandemia, brasil, segundo
LDA 3: como, pode, mas, ou, agua, energia, podem, sistema, diz, isso
LDA 4: como, mas, anos, us, esta, ja, ate, mil, ha, pela


## 5. Comparar com as editorias e encontrar confusão

A tabela abaixo é um diagnóstico, não uma nota de acerto: tópico e editoria são tipos diferentes de agrupamento. Leia ao menos duas notícias que ficaram em uma combinação inesperada.

In [6]:
tabela = pd.crosstab(base["topico_nmf"], base["editoria"])
print(tabela)
base["peso_topico"] = pesos.max(axis=1)
print("Menor confiança relativa:")
print(base.nsmallest(5, "peso_topico")[["id", "editoria", "topico_nmf", "peso_topico"]])

editoria    ciencia e tecnologia  economia  entretenimento  esportes  politica
topico_nmf                                                                    
0                             97         2               0         0         3
1                              0        86               3         0         6
2                              0         9               1         0        89
3                              1         0               4        95         1
4                              2         0              92         7         0
Menor confiança relativa:
                        id        editoria  topico_nmf  peso_topico
122        economia/31.txt        economia           2     0.038224
179        economia/83.txt        economia           1     0.039093
203  entretenimento/14.txt  entretenimento           4     0.039506
186         economia/9.txt        economia           1     0.046621
287  entretenimento/90.txt  entretenimento           4     0.046690


## 6. Versão com embeddings: BERTopic no Colab com GPU

Selecione **Ambiente de execução → Alterar tipo de ambiente → GPU**. BERTopic junta embeddings (representações numéricas de significado) e agrupamento. Primeiro confirme GPU; só então instale os pacotes. Usaremos 150 notícias, 30 de cada editoria, para não transformar a aula em uma espera. O resultado pode variar com versões e com o agrupamento; não precisa repetir os cinco tópicos do NMF.

In [7]:
import importlib.util
if importlib.util.find_spec("torch") is None:
    gpu = False
else:
    import torch
    gpu = torch.cuda.is_available()
print("GPU disponível:", gpu)
if not gpu:
    print("Ative GPU no Colab para executar o próximo bloco. O NMF e o LDA acima já estão completos.")

GPU disponível: False
Ative GPU no Colab para executar o próximo bloco. O NMF e o LDA acima já estão completos.


In [8]:
if gpu:
    import sys, subprocess, importlib.util
    if importlib.util.find_spec("bertopic") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bertopic", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer
    from bertopic import BERTopic
    amostra_gpu = base.groupby("editoria", group_keys=False).sample(n=30, random_state=42)
    modelo_embedding = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cuda")
    stopwords_rotulo = stopwords + ["nao", "como", "mas", "ele", "ela", "eu", "voce"]
    vetor_bertopic = CountVectorizer(stop_words=stopwords_rotulo, strip_accents="unicode", min_df=2)
    modelo_bertopic = BERTopic(embedding_model=modelo_embedding, vectorizer_model=vetor_bertopic,
                              min_topic_size=8, calculate_probabilities=False, verbose=False)
    topicos_gpu, _ = modelo_bertopic.fit_transform(amostra_gpu["texto_original"].tolist())
    print(modelo_bertopic.get_topic_info()[["Topic", "Count", "Name"]].head(10))

## 7. Quando der errado

- **Stopwords inconsistentes:** `strip_accents` remove acentos; ajuste a lista ou aceite que algumas palavras apareçam nos tópicos.
- **Tópicos genéricos:** leia os textos de maior peso, reduza termos muito frequentes e explique a mudança.
- **BERTopic demora na primeira vez:** ele baixa um modelo; execute somente a amostra indicada.
- **Sem GPU:** NMF e LDA continuam válidos. Ative GPU só para a comparação da seção 6.

**Missão:** abra o exercício e nomeie dois tópicos claros e um confuso com evidência das notícias.